# Create a training set from annotated data

- Load the data
- Filter by object classes
- Merge groups together spatially
- Create train/val/test split
- Copy data

In [ ]:
import json
import os
import shutil

from typing import Dict, List, Tuple

import geopandas as gpd
import pandas as pd

from xyzservices import TileProvider


RD_CRS = "EPSG:28992"  # CRS code for the Dutch Rijksdriehoek coordinate system
LAT_LON_CRS = "EPSG:4326"  # CRS code for WGS84 latitude/longitude coordinate system

ams_tile_provider = TileProvider(
    name="Topografie, standaard visualisatie (WM)",
    url="https://t1.data.amsterdam.nl/topo_wm/{z}/{x}/{y}.png",
    attribution="data.amsterdam.nl",
)


def load_json(file_name):
    with open(file_name, 'r') as file:
        return json.load(file)


def load_and_merge_annotations(coco_labels_file: str, metadata_file: str, categories: Dict[int, str], group_prefix: str) -> Tuple[pd.DataFrame, gpd.GeoDataFrame]:
    coco_data = load_json(coco_labels_file)
    coco_data["annotations"] = [
        annot for annot in coco_data["annotations"] 
        if annot["category_id"] in categories.keys()
    ]

    images_df = pd.DataFrame(data=coco_data["images"]).set_index("id")[["file_name", "width", "height"]]
    images_df.insert(
        loc=1, 
        column="name", 
        value=[os.path.splitext(os.path.basename(file))[0] for file in images_df["file_name"]]
    )

    metadata_gdf = gpd.read_file(metadata_file)[["file_name", "group", "geometry"]]
    metadata_gdf["name"] = [os.path.splitext(file_name)[0] for file_name in metadata_gdf["file_name"]]
    metadata_gdf.set_index("name", inplace=True)
    metadata_gdf.drop(columns="file_name", inplace=True)
    metadata_gdf["group"] = group_prefix + metadata_gdf["group"].astype(str)

    annotations_df = pd.DataFrame(data=coco_data["annotations"]).set_index("id")

    annotations_df[["x_min", "y_min", "x_width", "y_height"]] = annotations_df["bbox"].apply(lambda row: pd.Series(data=row))
    annotations_df["area"] = annotations_df["x_width"] * annotations_df["y_height"]

    annotations_df = (
        annotations_df
        .join(images_df, on="image_id", how="left")
        .join(metadata_gdf, on="name", how="left")
    )

    return annotations_df, metadata_gdf


def count_categories(image_df: pd.DataFrame, categories: Dict[int, str]) -> pd.Series:
    counts = image_df["category_id"].value_counts()

    data = {
        categories[key]: counts.loc[key]
        if key in counts.index
        else 0
        for key in sorted(categories.keys())
    }

    return pd.Series(
        data
    )


def coco_bbox_to_yolo(coco_bbox: List[float], img_width: int, img_height: int) -> List[float]:
    """
    Converts COCO bounding box format to YOLO format.

    Parameters
    ----------
    bbox : List[float]
        Bounding box in COCO format [x_min, y_min, width, height].
    img_width : int
        Width of the image.
    img_height : int
        Height of the image.

    Returns
    -------
    List[float]
        Bounding box in YOLO format [x_center, y_center, width, height], normalized.
    """
    x_min, y_min, width, height = coco_bbox
    try:
        x_center = x_min + width / 2
        y_center = y_min + height / 2

        return [x_center, y_center, width, height]
    except ZeroDivisionError:
        print(f"ZeroDivisionError encountered with: img_width={img_width}, img_height={img_height}, x_min={x_min}, y_min={y_min}, bbox_width={width}, bbox_height={height}")
        return None


def write_labels(dataset: pd.DataFrame, annotations_df: pd.DataFrame, class_map: Dict[int, int], output_folder: str):
    os.makedirs(output_folder, exist_ok=True)

    for name in dataset.index.values:
        yolo_string = ""
        for row in annotations_df[annotations_df["name"]==name].itertuples():
            yolo_bbox = coco_bbox_to_yolo(row.bbox, row.width, row.height)
            yolo_string += f"{class_map[row.category_id]} {' '.join(map(str, yolo_bbox))}\n"
        
        file_path = os.path.join(output_folder, f"{name}.txt")
        with open(file_path, "w") as f:
            f.write(yolo_string)


images_folder = {
    "250514": "../datasets/experiments/zwerfafval/annotatieproject/inwinning_250514_selectie_300",
    "260421": "../datasets/experiments/zwerfafval/annotatieproject/inwinning_260422_selectie_1000",  # NOTE: typo is on purpose
}

def copy_images(dataset: pd.DataFrame, output_folder: str):
    os.makedirs(output_folder, exist_ok=True)

    for row in dataset.itertuples():
        prefix = row.group.split(sep="_")[0]
        source_file = os.path.join(images_folder[prefix], f"{row.Index}.jpg")
        shutil.copy2(source_file, output_folder)


def copy_background_images(dataset: pd.DataFrame, metadata_df: pd.DataFrame, output_folder: str):
    os.makedirs(output_folder, exist_ok=True)
    
    for group in dataset["group"].unique():
        all_names = set(metadata_df[metadata_df["group"]==group].index.values)
        background_names = all_names - set(dataset[dataset["group"]==group].index.values)
        
        for name in background_names:
            prefix = group.split(sep="_")[0]
            source_file = os.path.join(images_folder[prefix], f"{name}.jpg")
            shutil.copy2(source_file, output_folder)

## Load annotations and metadata

In [ ]:
categories = {
    1: "zwerfafval_grof",
    3: "grofvuil",
    4: "zwerfafval_fijn"
}

In [ ]:
# Load annotations and metadata for the data collection of 14-05-2025 (De Pijp)

# label_file = "../datasets/experiments/zwerfafval/labels/labels_250514_300_v0.1.json"
label_file = "../datasets/experiments/zwerfafval/labels/zwerfafval_250514_300_v1.json"
metadata_file = "../datasets/experiments/zwerfafval/recording_2025-05-14_19-47-40/dataselectie_300.gpkg"

annotations_250514_df, metadata_250514_gdf = load_and_merge_annotations(label_file, metadata_file, categories, group_prefix="250514_")

In [ ]:
# Load annotations and metadata for the data collection of 21-04-2026 (West)

# label_file = "../datasets/experiments/zwerfafval/labels/labels_260421_1000_v0.1.json"
label_file = "../datasets/experiments/zwerfafval/labels/zwerfafval_260421_712_v1.json"
metadata_file = "../datasets/experiments/zwerfafval/inwinning_260421_selectie_1000_v1.gpkg"

annotations_260421_df, metadata_260421_gdf = load_and_merge_annotations(label_file, metadata_file, categories, group_prefix="260421_")

In [ ]:
# Merge into one set

annotations_df = pd.concat([annotations_250514_df, annotations_260421_df])
metadata_gdf = pd.concat([metadata_250514_gdf, metadata_260421_gdf])
# annotations_df = annotations_250514_df
# metadata_gdf = metadata_250514_gdf

In [ ]:
annotations_df

In [ ]:
# Scatter plot of bounding box coordinates
annotations_df[annotations_df["category_id"]==1].plot.scatter(x="x_min", y="y_min", ylim=[1, 0], c="area")

In [ ]:
# Scatter plot of bounding box size
annotations_df[annotations_df["category_id"]==1].plot.scatter(x="x_width", y="y_height")

In [ ]:
# Show largest bounding boxes
annotations_df.sort_values(by="area").tail(10)

In [ ]:
# Count annotations per object class
count_categories(annotations_df, categories=categories)

## Merge groups

Merge groups when they are close to each other, to prevent leakage between dataset splits

In [ ]:
from typing import Set, Union

from shapely.geometry import LineString, Point

def to_linestring(gdf: gpd.GeoDataFrame) -> Union[LineString, Point]:
    if len(gdf) != 1:
        return LineString(gdf.geometry)
    else:
        return gdf["geometry"].iloc[0]
    

def groups_within_distance(row: pd.Series, distance_threshold: float = 10.0, checked_set: Set[str] = set()) -> Set[str]:
    # print(f"Checking {row.name}..")
    group_set = set(
        group_linestrings_gdf[
            group_linestrings_gdf.distance(row["geometry"]) <= distance_threshold
        ].index.values
    )
    # print(group_set)
    if len(group_set) == 1:
        # print("Done")
        return group_set
    else:
        checked_set = checked_set.union({row.name})
        # print(f"Checked: {checked_set}")
        other_groups = list(group_set - checked_set)
        if len(other_groups) == 0:
            # print("Done")
            return checked_set
        else:
            # print(f"Will check {other_groups}..")
            other_groups_sets = (
                group_linestrings_gdf
                .loc[other_groups]
                .apply(groups_within_distance, args=(distance_threshold, checked_set), axis=1)
                .values
            )
            return checked_set.union(*other_groups_sets)


distance_threshold = 20  # How close should groups be to each other to be merged
    
groups_gdf = (
    metadata_gdf
    .loc[annotations_df["name"].unique(), :]
    .to_crs(RD_CRS)
    .sort_index()
)

group_linestrings = groups_gdf.groupby(by="group").apply(to_linestring)
group_linestrings_gdf = gpd.GeoDataFrame(geometry=group_linestrings, index=group_linestrings.index, crs=RD_CRS)

group_linestrings_gdf["grouped_sets"] = group_linestrings_gdf.apply(groups_within_distance, args=(distance_threshold, set()), axis=1)
group_linestrings_gdf["new_group"] = group_linestrings_gdf["grouped_sets"].apply(lambda row: "-".join(row))

annotations_df["new_group"] = annotations_df["group"].map(group_linestrings_gdf["new_group"])

## Generate dataset with train/val/test split

We use the method GroupShuffleSplit from the scikit-learn package. This method can split data while keeping pre-defined groups intact. This ensures the consecutive images (with visual overlap) fall into a single data split.

In [ ]:
# GroupShuffleSplit requires a dataset that has one row per image, with columns
# showing the count of each object class, and one additional group tag

dataset = annotations_df.groupby(by="name", group_keys=True).apply(func=count_categories, categories=categories)
dataset["group"] = (
    annotations_df
    .groupby(by="name", group_keys=True)
    .apply(lambda df: df["new_group"].iloc[0])
    .loc[dataset.index]
)

dataset

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

classes = list(categories.values())  # List of class names
train_proportion = 0.7  # Fraction of dataset to go into train split
split_val_test = False  # Whether to split the remainder in val/test

splitter_train_val = GroupShuffleSplit(n_splits=1, test_size=(1-train_proportion))

train_ix, not_train_ix = next(
    splitter_train_val.split(
        X=dataset[["group"]], 
        y=dataset[classes], 
        groups=dataset["group"],
    )
)

if split_val_test:
    # Separate the remaining data (not in the train split)
    not_train_data = dataset.iloc[not_train_ix, :]

    splitter_test = GroupShuffleSplit(n_splits=1, test_size=0.5)  # Val en test are split evenly in the remaining data

    val_ix, test_ix = next(
        splitter_test.split(
            X=not_train_data[["group"]], 
            y=not_train_data[classes], 
            groups=not_train_data["group"],
        )
    )

if not split_val_test:
    not_train_data = dataset
    val_ix = not_train_ix
    test_ix = []

train_distribution = dataset[classes].iloc[train_ix, :].sum()
val_distribution = not_train_data[classes].iloc[val_ix, :].sum()
test_distribution = not_train_data[classes].iloc[test_ix, :].sum()

dataset_train = dataset.iloc[train_ix, :]
dataset_val = not_train_data.iloc[val_ix, :]
dataset_test = not_train_data.iloc[test_ix, :]


print(f"Train fraction: {len(train_ix) / len(dataset)}")
print(f"Validation fraction: {len(val_ix) / len(dataset)}")
print(f"Test fraction: {len(test_ix) / len(dataset)}")
print("-----")
print("Training set")
print(train_distribution / dataset[classes].sum())
print("\nValidation set:")
print(val_distribution / dataset[classes].sum())
print("\nTest set:")
print(test_distribution / dataset[classes].sum())

In [ ]:
metadata_gdf["split"] = "none"
metadata_gdf.loc[dataset_train.index.values, "split"] = "train"
metadata_gdf.loc[dataset_val.index.values, "split"] = "val"
metadata_gdf.loc[dataset_test.index.values, "split"] = "test"

import matplotlib.colors as colors
if split_val_test:
    my_cmap = colors.ListedColormap(["lightgray", "orange", "green", "yellow"])
    my_cmap_splits_only = colors.ListedColormap(["orange", "green", "yellow"])
else:
    my_cmap = colors.ListedColormap(["lightgray", "green", "yellow"])
    my_cmap_splits_only = colors.ListedColormap(["green", "yellow"])

In [ ]:
metadata_gdf.to_crs(RD_CRS).plot(column="split", cmap=my_cmap, legend=True)

In [ ]:
split_map = metadata_gdf.to_crs(RD_CRS).explore(column="split", cmap=my_cmap, legend=True, tiles=ams_tile_provider)
# map = metadata_gdf[metadata_gdf["split"]!="none"].to_crs(RD_CRS).explore(column="split", cmap=my_cmap_splits_only, legend=True, tiles=ams_tile_provider)

split_map.save(os.path.join("../datasets/experiments/zwerfafval", "dataset_v1_split.html"))

## Copy images and generate YOLO annotations

In [ ]:
dataset_folder = "../datasets/experiments/zwerfafval/dataset_v1"

annotations_output_folder = os.path.join(dataset_folder, "labels")
images_output_folder = os.path.join(dataset_folder, "images")
background_output_folder = os.path.join(dataset_folder, "background_images")

dataset_classes = {
    0: "zwerfafval_fijn",
    1: "zwerfafval_grof",
    2: "grofvuil",
}

cat_inv = {v: k for k, v in categories.items()}
class_mapping = {cat_inv[cat_name]: new_id for new_id, cat_name in dataset_classes.items()}

dataset_parts: Dict[str, pd.DataFrame] = {
    "train": dataset_train,
    "val": dataset_val,
    "test": dataset_test,
}

for (part, part_dataset) in dataset_parts.items():
    write_labels(
        dataset=part_dataset,
        annotations_df=annotations_df,
        class_map=class_mapping,
        output_folder=os.path.join(annotations_output_folder, part)
    )

    copy_images(
        dataset=part_dataset,
        output_folder=os.path.join(images_output_folder, part)
    )

    copy_background_images(
        dataset=part_dataset,
        metadata_df=metadata_gdf,
        output_folder=os.path.join(background_output_folder, part)
    )

In [ ]:
# TODO

# manually select X background images per split
# add bg to splits